# EMPIRE → nodal disaggregation — result maps

Interactive folium maps of what `empire_nodal.jl` did with the EMPIRE expansion
results (scenario read from `config.toml → [scenario]`):

1. **Regions** — NUTS3 choropleth of the capacity change 2024 → scenario, with a
   per-tech breakdown table in each region's popup.
2. **Generation fleet** — every unit on the map, switchable between the **2024
   fleet** and the **scenario fleet** (scaled + greenfield units, retired units
   hollow); Li-Ion BESS sites as an optional overlay.
3. **Grid** — every line, switchable between the **2024 grid** and the **updated
   grid** (corridor-reinforced lines coloured by expansion factor, synthetic new
   corridors dashed purple); the EMPIRE NUTS3 corridor layer as an optional overlay.

**Prerequisite:** `julia --project=. empire_nodal.jl` must have been run for the
active scenario (it writes `results/<label>/nodal/*.csv`).

Each map is saved to `results/<label>/nodal/map_*.html` and **opened in the
default external browser** (flag `OPEN_IN_BROWSER` below; set `SHOW_INLINE = True`
to also render them inside the notebook — that adds ~5 MB per map to the file).


In [1]:
import json
import numpy as np
import pandas as pd
import folium
import branca.colormap as bcm
from pathlib import Path

try:
    import tomllib as toml_loader          # Python 3.11+
except ModuleNotFoundError:
    import tomli as toml_loader

with open('config.toml', 'rb') as f:
    _cfg = toml_loader.load(f)
LABEL  = _cfg['scenario']['label']
PERIOD = int(_cfg['scenario']['period'])
NODAL  = Path('results') / LABEL / 'nodal'
assert NODAL.exists(), f'{NODAL} missing — run: julia --project=. empire_nodal.jl'

DATA     = Path('Data')
bus      = pd.read_csv(DATA / 'Bus_Data.csv').set_index('bus_id')      # y=lat, x=lon
bus_nuts = pd.read_csv(DATA / 'bus_nuts3.csv').set_index('bus_id')['nuts3']
lines    = pd.read_csv(DATA / 'lines.csv')
gens     = pd.read_csv(DATA / 'generations.csv').set_index('unit_id')
nuts3    = json.load(open(DATA / 'nuts3_es.geojson', encoding='utf-8'))

units     = pd.read_csv(NODAL / 'unit_scale.csv')       # existing fleet + factors
new_units = pd.read_csv(NODAL / 'new_units.csv')        # greenfield units
regions_t = pd.read_csv(NODAL / 'gen_regions.csv')      # per (tech, region) targets
line_scl  = pd.read_csv(NODAL / 'line_scale.csv').set_index('line_id')['factor']
new_lines = pd.read_csv(NODAL / 'new_lines.csv')
corridors = pd.read_csv(NODAL / 'corridors.csv').rename(columns={'from': 'r0', 'to': 'r1'})
bess      = pd.read_csv(NODAL / 'bess_units.csv')

# unit coordinates: the plant site from generations.csv (fallback: its bus)
units = units.join(gens[['y', 'x']], on='unit_id')
units['y'] = units['y'].fillna(units['bus_id'].map(bus['y']))
units['x'] = units['x'].fillna(units['bus_id'].map(bus['x']))
for _df in (new_units, bess):
    if len(_df):
        _df['y'] = _df['bus_id'].map(bus['y'])
        _df['x'] = _df['bus_id'].map(bus['x'])

# line endpoints + nameplate rating √3·V·Imax·circuits [MW] + expansion factor
for c in ('0', '1'):
    lines['y' + c] = lines['bus' + c].map(bus['y'])
    lines['x' + c] = lines['bus' + c].map(bus['x'])
lines['mw']     = np.sqrt(3) * lines['voltage'] * lines['Imax'] * lines['circuits'].clip(lower=1)
lines['factor'] = lines['line_id'].map(line_scl).fillna(1.0)
lines = lines.dropna(subset=['y0', 'x0', 'y1', 'x1'])

# region centroids = mean of member-bus coordinates (for the corridor overlay)
cent = bus.assign(region=bus_nuts).groupby('region')[['y', 'x']].mean()

# repo-wide fuel colours (same as results_plots*.ipynb) + scenario-only fuels
FUEL_COLORS = {
    'Wind':    '#4CAF50',
    'Solar':   '#FFC107',
    'Hydro':   '#2196F3',
    'Nuclear': '#9C27B0',
    'Gas':     '#F44336',
    'Coal':    '#212121',
    'Biomass': '#795548',
    'Oil':     '#607D8B',
    'Waste':   '#827717',
}
fuel_color = lambda f: FUEL_COLORS.get(f, '#9E9E9E')

print(f'scenario {LABEL} (period {PERIOD}): {len(units)} scaled units | '
      f'{len(new_units)} new units | {len(line_scl)} reinforced lines | '
      f'{len(new_lines)} new corridors | {len(bess)} BESS sites')


scenario GoRES (period 2): 958 scaled units | 169 new units | 3 reinforced lines | 2 new corridors | 94 BESS sites


In [2]:
# ── map helpers ──────────────────────────────────────────────
import webbrowser

OPEN_IN_BROWSER = True    # pop each saved map in the default browser
SHOW_INLINE     = False   # also render the maps inside the notebook (heavy)

ES_CENTER = (40.2, -3.6)

def base_map(zoom=6):
    # tiles added with control=False so the layer control only carries the
    # 2024/scenario radio groups and the optional overlays
    m = folium.Map(location=ES_CENTER, zoom_start=zoom, tiles=None, control_scale=True)
    folium.TileLayer('cartodbpositron', control=False).add_to(m)
    return m

def popup_html(title, rows):
    tr = ''.join(f"<tr><td style='padding:1px 10px 1px 0;color:#555;"
                 f"white-space:nowrap'>{k}</td>"
                 f"<td style='text-align:right;white-space:nowrap'><b>{v}</b></td></tr>"
                 for k, v in rows)
    return folium.Popup(f"<div style='font-family:sans-serif;font-size:12px'>"
                        f"<b>{title}</b><table>{tr}</table></div>", max_width=340)

def add_legend(m, title, entries):
    items = ''.join(
        f"<div><span style='display:inline-block;width:10px;height:10px;"
        f"border-radius:50%;background:{c};margin-right:6px'></span>{l}</div>"
        for c, l in entries)
    m.get_root().html.add_child(folium.Element(
        f"<div style='position:fixed;bottom:24px;left:12px;z-index:9999;"
        f"background:rgba(255,255,255,.92);padding:8px 12px;border-radius:6px;"
        f"font-family:sans-serif;font-size:12px;line-height:1.5;"
        f"box-shadow:0 1px 4px rgba(0,0,0,.3)'><b>{title}</b>{items}</div>"))

def save(m, name):
    out = (NODAL / name).resolve()
    m.save(str(out))
    print('saved', out)
    if OPEN_IN_BROWSER:
        webbrowser.open(out.as_uri())
    return m if SHOW_INLINE else None


## 1 · Regional overview — where EMPIRE builds and retires

Choropleth of the net installed-capacity change per NUTS3 region (scenario target
vs the 2024 fleet mapped into that region).  Diverging scale: red = net
retirement, blue = net build-out, gray ≈ unchanged.  Click a region for its
per-tech table: 2024 MW → target MW (scale factor), greenfield MW, and the
region's BESS power.


In [3]:
reg = (regions_t.groupby('region')
       .agg(existing=('existing_mw', 'sum'), target=('target_mw', 'sum'),
            new_mw=('new_mw', 'sum'))
       .assign(delta=lambda d: d.target - d.existing))
bess_reg = (bess.assign(region=bess['bus_id'].map(bus_nuts))
            .groupby('region')['power_mw'].sum()) if len(bess) else pd.Series(dtype=float)

vmax = float(reg['delta'].abs().max())
cmap = bcm.LinearColormap(['#B2182B', '#F7F7F7', '#2166AC'], vmin=-vmax, vmax=vmax,
                          caption='Δ installed capacity 2024 → scenario [MW]')

m1 = base_map()
for feat in nuts3['features']:
    rid = feat['properties']['NUTS_ID']
    if rid not in reg.index:
        continue
    r    = reg.loc[rid]
    sub  = regions_t[regions_t.region == rid].sort_values('target_mw', ascending=False)
    rows = [(t.tech, f'{t.existing_mw:,.0f} → {t.target_mw:,.0f} MW (×{t.factor:.2f})'
             + (f' | +{t.new_mw:,.0f} new' if t.new_mw > 0 else ''))
            for t in sub.itertuples() if max(t.existing_mw, t.target_mw) > 1]
    if bess_reg.get(rid, 0) > 1:
        rows.append(('Li-Ion BESS', f'{bess_reg[rid]:,.0f} MW'))
    rows.append(('Net change', f'{r.delta:+,.0f} MW'))
    gj = folium.GeoJson(
        feat,
        style_function=lambda f, d=float(r.delta): {
            'fillColor': cmap(d), 'fillOpacity': 0.75, 'color': '#666', 'weight': 0.7},
        highlight_function=lambda f: {'weight': 2, 'color': '#333'})
    gj.add_child(popup_html(f"{rid} — {feat['properties']['NUTS_NAME']}", rows))
    gj.add_to(m1)
cmap.add_to(m1)
save(m1, 'map_regions.html')


saved C:\Users\ehsanno\DataspellProjects\Spanish_Power_System\results\GoRES\nodal\map_regions.html


## 2 · Generation fleet — 2024 vs scenario

Radio toggle between the **2024 fleet** and the **scenario fleet**.  Marker area ∝
capacity, colour = fuel.  In the scenario view, greenfield units carry a bold dark
ring, retired units shrink to hollow gray dots.  The Li-Ion BESS overlay (teal) can
be switched on independently.  Every marker's popup has the unit's full record,
including its nodal scale factor.


In [4]:
m2   = base_map()
fg24 = folium.FeatureGroup(name='Fleet 2024', overlay=False, show=False).add_to(m2)
fgsc = folium.FeatureGroup(name=f'Fleet {LABEL}', overlay=False, show=True).add_to(m2)
fgbe = folium.FeatureGroup(name='Li-Ion BESS (scenario)', overlay=True, show=False).add_to(m2)

rad = lambda mw: max(2.0, 0.35 * np.sqrt(max(mw, 0.0)))

for u in units.itertuples():
    rows = [('Fuel / tech', f'{u.primary_fuel} / {u.technology}'),
            ('Bus', u.bus_id), ('Region', u.region),
            ('2024', f'{u.cap_2024_mw:,.0f} MW'),
            ('Nodal factor', f'×{u.factor:.2f}'),
            (LABEL, f'{u.cap_scen_mw:,.0f} MW')]
    c = fuel_color(u.primary_fuel)
    folium.CircleMarker([u.y, u.x], radius=rad(u.cap_2024_mw), color=c, weight=1,
                        fill=True, fill_color=c, fill_opacity=0.65,
                        popup=popup_html(u.unit_id, rows)).add_to(fg24)
    if u.cap_scen_mw > 1:
        folium.CircleMarker([u.y, u.x], radius=rad(u.cap_scen_mw), color=c, weight=1,
                            fill=True, fill_color=c, fill_opacity=0.65,
                            popup=popup_html(u.unit_id, rows)).add_to(fgsc)
    else:
        folium.CircleMarker([u.y, u.x], radius=2.5, color='#9E9E9E', weight=1.2,
                            fill=False,
                            popup=popup_html(f'{u.unit_id} — retired', rows)).add_to(fgsc)

for u in new_units.itertuples():
    rows = [('Fuel / tech', f'{u.primary_fuel} / {u.technology}'),
            ('Bus', u.bus_id), ('Capacity', f'{u.capacity_mw:,.0f} MW'),
            ('Origin', 'greenfield (EMPIRE build)')]
    folium.CircleMarker([u.y, u.x], radius=rad(u.capacity_mw), color='#212121',
                        weight=2.5, fill=True, fill_color=fuel_color(u.primary_fuel),
                        fill_opacity=0.9,
                        popup=popup_html(f'{u.unit_id} — new', rows)).add_to(fgsc)

for b in bess.itertuples():
    rows = [('Power', f'{b.power_mw:,.0f} MW'), ('Energy', f'{b.energy_mwh:,.0f} MWh'),
            ('Duration', f'{b.energy_mwh / max(b.power_mw, 1e-9):.1f} h')]
    folium.CircleMarker([b.y, b.x], radius=rad(b.power_mw), color='#00695C',
                        weight=1.5, fill=True, fill_color='#26A69A', fill_opacity=0.8,
                        popup=popup_html(f'BESS @ {b.bus_id}', rows)).add_to(fgbe)

folium.LayerControl(collapsed=False).add_to(m2)
add_legend(m2, 'Fuel',
           [(c, f) for f, c in FUEL_COLORS.items()] + [('#26A69A', 'Li-Ion BESS')])
save(m2, 'map_fleet.html')


saved C:\Users\ehsanno\DataspellProjects\Spanish_Power_System\results\GoRES\nodal\map_fleet.html


## 3 · Grid — 2024 vs updated (corridor reinforcement)

Radio toggle between the **2024 grid** and the **updated grid**.  Line width tracks
voltage (400 kV thicker), HVDC is dashed.  In the updated view, lines scaled by an
EMPIRE corridor investment are coloured by their expansion factor (orange ramp) and
drawn wider; synthetic lines for brand-new corridors are dashed purple.  The
optional **EMPIRE NUTS3 corridors** overlay draws every intra-ES corridor of the
investment model centroid-to-centroid — blue where EMPIRE invested, gray otherwise —
with the initial/installed capacities and the matching note in the popup.

The **cross-border corridors** overlay (on by default, dashed) shows the EMPIRE
interconnections from Spanish NUTS3 nodes to France / Portugal / Italy — blue where
EMPIRE expanded them (GoRES: +2 GW Navarra→FR, +2 GW Barcelona→IT, +2 GW
Valencia→IT).  These enter the market chain as **zonal NTCs** (Italy folds into the
EU zone); the bus-grid redispatch keeps the fixed `crossborder.csv` border
injections, so they are drawn schematically, not as physical lines.


In [5]:
m3   = base_map()
g24  = folium.FeatureGroup(name='Grid 2024', overlay=False, show=False).add_to(m3)
gsc  = folium.FeatureGroup(name=f'Grid {LABEL} (updated)', overlay=False, show=True).add_to(m3)
gco  = folium.FeatureGroup(name='EMPIRE NUTS3 corridors', overlay=True, show=False).add_to(m3)

fmax  = max(float(lines['factor'].max()), 1.5)
fcmap = bcm.LinearColormap(['#FDAE6B', '#8C2D04'], vmin=1.0, vmax=fmax,
                           caption='corridor expansion factor (updated grid)')

for L in lines.itertuples():
    seg  = [(L.y0, L.x0), (L.y1, L.x1)]
    dash = '6 6' if L.dc == 't' else None
    w    = 2.2 if L.voltage >= 380 else 1.1
    rows = [('Voltage', f'{L.voltage:.0f} kV'), ('Circuits', f'{L.circuits:.0f}'),
            ('Type', 'HVDC' if L.dc == 't' else 'AC'),
            ('Nameplate', f'{L.mw:,.0f} MW')]
    folium.PolyLine(seg, color='#9AA0A6', weight=w, opacity=0.8, dash_array=dash,
                    popup=popup_html(L.line_id, rows)).add_to(g24)
    if L.factor > 1.0 + 1e-6:
        rows2 = rows + [('Expansion', f'×{L.factor:.2f}'),
                        ('New nameplate', f'{L.mw * L.factor:,.0f} MW')]
        folium.PolyLine(seg, color=fcmap(L.factor), weight=4.0, opacity=0.95,
                        dash_array=dash,
                        popup=popup_html(f'{L.line_id} — reinforced', rows2)).add_to(gsc)
    else:
        folium.PolyLine(seg, color='#9AA0A6', weight=w, opacity=0.55, dash_array=dash,
                        popup=popup_html(L.line_id, rows)).add_to(gsc)

for L in new_lines.itertuples():
    p0, p1 = bus.loc[L.bus0], bus.loc[L.bus1]
    rows = [('Route', f'{L.bus0} → {L.bus1}'), ('Voltage', f'{L.voltage:.0f} kV'),
            ('Length', f'{L.length:,.0f} km'),
            ('Capacity', f'{np.sqrt(3) * L.voltage * L.Imax:,.0f} MW'),
            ('Origin', 'synthetic (new EMPIRE corridor)')]
    folium.PolyLine([(p0.y, p0.x), (p1.y, p1.x)], color='#7B1FA2', weight=3.2,
                    dash_array='10 6',
                    popup=popup_html(f'{L.line_id} — new corridor', rows)).add_to(gsc)

for C in corridors.query("kind == 'intra'").itertuples():
    if C.r0 not in cent.index or C.r1 not in cent.index:
        continue
    invested = C.installed_mw > C.initial_mw + 1.0
    rows = [('Initial', f'{C.initial_mw:,.0f} MW'),
            ('Installed (P{})'.format(PERIOD), f'{C.installed_mw:,.0f} MW'),
            ('Factor', '-' if pd.isna(C.factor) else f'×{C.factor:.2f}'),
            ('Physical lines', f'{C.n_lines}')]
    if isinstance(C.note, str) and C.note:
        rows.append(('Note', C.note))
    folium.PolyLine([tuple(cent.loc[C.r0]), tuple(cent.loc[C.r1])],
                    color='#1565C0' if invested else '#90A4AE',
                    weight=float(np.clip(C.installed_mw / 2500, 1.0, 6.0)),
                    opacity=0.85 if invested else 0.45,
                    popup=popup_html(f'{C.r0} — {C.r1}', rows)).add_to(gco)

# cross-border EMPIRE corridors (schematic: NUTS3 centroid → country anchor)
XB_ANCHOR = {'France': (44.9, 1.4), 'Portugal': (39.6, -8.1), 'Italy': (40.8, 8.4)}
gxb = folium.FeatureGroup(name='Cross-border corridors (EMPIRE)',
                          overlay=True, show=True).add_to(m3)
for C in corridors.query("kind == 'xborder'").itertuples():
    if C.r0 not in cent.index or C.r1 not in XB_ANCHOR:
        continue
    added = C.installed_mw - C.initial_mw
    rows  = [('Initial', f'{C.initial_mw:,.0f} MW'),
             ('Installed (P{})'.format(PERIOD), f'{C.installed_mw:,.0f} MW'),
             ('Added', f'{added:+,.0f} MW'),
             ('In the chain', 'zonal NTC (Italy → EU zone)' if C.r1 == 'Italy'
                              else 'zonal NTC')]
    folium.PolyLine([tuple(cent.loc[C.r0]), XB_ANCHOR[C.r1]],
                    color='#1565C0' if added > 1 else '#78909C',
                    weight=float(np.clip(C.installed_mw / 1500, 1.5, 6.0)),
                    opacity=0.9 if added > 1 else 0.5, dash_array='4 8',
                    popup=popup_html(f'{C.r0} — {C.r1}', rows)).add_to(gxb)

fcmap.add_to(m3)
folium.LayerControl(collapsed=False).add_to(m3)
add_legend(m3, 'Grid',
           [('#9AA0A6', 'unchanged line'), ('#C05014', 'reinforced (orange ramp)'),
            ('#7B1FA2', 'new corridor (dashed)'),
            ('#1565C0', 'EMPIRE corridor, invested'),
            ('#78909C', 'cross-border, unchanged (dashed)')])
save(m3, 'map_grid.html')


saved C:\Users\ehsanno\DataspellProjects\Spanish_Power_System\results\GoRES\nodal\map_grid.html


## 4 · Numbers behind the maps


In [6]:
print(f'=== {LABEL} (period {PERIOD}) — disaggregation summary ===')

print('\nTop-10 regions by net capacity change [MW]:')
display(reg.sort_values('delta', ascending=False).head(10).round(0))

if len(new_units):
    print('Greenfield units by (fuel, technology):')
    display(new_units.groupby(['primary_fuel', 'technology'])
            .agg(units=('unit_id', 'count'), mw=('capacity_mw', 'sum')).round(0))

intra = corridors.query("kind == 'intra'")
print('Reinforced / new intra-ES corridors:')
display(intra[intra.installed_mw > intra.initial_mw + 1.0]
        .sort_values('installed_mw', ascending=False).round(1))

print('Cross-border corridors (zonal NTC in the chain):')
display(corridors.query("kind == 'xborder'")
        .assign(added_mw=lambda d: d.installed_mw - d.initial_mw)
        [['r0', 'r1', 'initial_mw', 'installed_mw', 'added_mw']]
        .sort_values('added_mw', ascending=False).round(1))

print('Intra-ES corridors the mapping could not fully resolve (see note):')
display(intra[intra.note.notna() & (intra.note != '')]
        [['r0', 'r1', 'initial_mw', 'installed_mw', 'n_lines', 'note']].round(1))

if len(bess):
    print('BESS by region [MW]:')
    display(bess.assign(region=bess['bus_id'].map(bus_nuts))
            .groupby('region')[['power_mw', 'energy_mwh']].sum()
            .sort_values('power_mw', ascending=False).round(0).head(10))


=== GoRES (period 2) — disaggregation summary ===

Top-10 regions by net capacity change [MW]:


,existing,target,new_mw,delta
region,,,,
ES243,7230.0,21939.0,368.0,14708.0
ES412,1944.0,10308.0,64.0,8365.0
ES513,555.0,7844.0,0.0,7289.0
ES242,2381.0,7948.0,69.0,5567.0
ES612,7468.0,12735.0,0.0,5267.0
ES423,3497.0,8250.0,157.0,4753.0
ES521,282.0,4738.0,13.0,4456.0
ES241,860.0,4524.0,33.0,3664.0
ES611,1356.0,4744.0,67.0,3388.0


Greenfield units by (fuel, technology):


,,units,mw
primary_fuel,technology,,
Biomass,BioCCS,15,2631.0
Gas,Gas_turbine,124,4271.0
Solar,PV,3,37.0
Waste,Waste,27,714.0


Reinforced / new intra-ES corridors:


...

,r0,r1,initial_mw,installed_mw,added_mw
69,ES523,Italy,0.0,2000.0,2000.0
70,ES511,Italy,0.0,2000.0,2000.0
72,ES220,France,0.0,2000.0,2000.0
11,ES113,Portugal,8666.9,8666.9,0.0
58,ES512,France,2860.0,2860.0,0.0
64,ES212,France,2290.2,2290.2,0.0
71,ES213,France,2000.0,2000.0,0.0
85,ES432,Portugal,1787.5,1787.5,0.0
86,ES431,Portugal,1787.5,1787.5,0.0
87,ES615,Portugal,1787.5,1787.5,0.0


Intra-ES corridors the mapping could not fully resolve (see note):


,r0,r1,initial_mw,installed_mw,n_lines,note
51,ES243,ES424,3575.0,3575.0,2,border mismatch: applied to ES417-ES424 lines
65,ES130,ES413,2279.0,2279.0,0,no physical line found between regions
67,ES422,ES432,2140.0,2140.0,0,no physical line found between regions
68,ES242,ES523,2030.0,2030.0,2,border mismatch: applied to ES242-ES514 lines
73,ES421,ES422,1970.0,1970.0,0,no physical line found between regions
75,ES422,ES618,1920.0,1920.0,0,no physical line found between regions
76,ES523,ES532,400.0,1900.0,1,border mismatch: applied to ES522-ES532 lines
89,ES111,ES113,1513.0,1513.0,0,no physical line found between regions
92,ES421,ES521,1440.0,1440.0,0,no physical line found between regions
98,ES412,ES418,0.0,514.3,0,synthetic line ES00691 - ES00089 (173.0 km)


BESS by region [MW]:


,power_mw,energy_mwh
region,,
ES511,4129.0,8259.0
ES300,3841.0,7683.0
ES521,1535.0,3070.0
ES611,1329.0,2659.0
ES614,1056.0,2112.0
ES512,516.0,1032.0
ES416,446.0,891.0
ES212,411.0,821.0
ES418,170.0,340.0
